# 08 FFN and SwiGLU

## Problem

为什么 Transformer block 里除了 attention 之外，还一定要有 FFN？SwiGLU 又到底改进了什么，它为什么会在很多现代模型里出现？

## Dependencies

建议先理解 residual、norm 与基础 attention。


## Goals

- 理解 FFN 是逐 token 独立工作的非线性子层
- 理解为什么仅有 attention 还不够
- 理解 ReLU FFN 和 SwiGLU FFN 的核心差别
- 能解释“门控”在这里到底是什么意思

## Scope and Approach

这一节重点不是背公式，而是讲清楚 FFN 在整个 block 里的角色。很多人知道 Transformer 有 attention 和 FFN 两部分，但不知道为什么 attention 之后还必须要有一个逐位置的非线性子层。这一节就把这件事讲透。


## 为什么 attention 之后还不够

attention 擅长做的是“看别人”：

- 当前位置可以参考别的位置
- 上下文信息可以被动态汇总回来

但仅有 attention 还不够，因为 attention 更像是在做**跨位置的信息交换**。模型还需要一个子层，去做：

- 每个位置内部的特征加工
- 更强的非线性变换
- 更复杂的组合特征表达

FFN 的角色，就是在每个 token 内部再“想一遍”。


## FFN 到底在做什么

最经典的 FFN 可以看成两步：

1. 先把 hidden dimension 扩大
2. 再经过非线性后压回原维度

它和 attention 的一个根本区别是：

- attention 会让不同位置互相通信
- FFN 通常对每个位置独立作用

也就是说，FFN 不负责“看别人”，它负责“把我自己这个位置的表示再加工一层”。


In [ ]:
import numpy as np

np.random.seed(7)
np.set_printoptions(precision=3, suppress=True)

# 这里构造一个很小的输入序列。
# shape = (seq_len, hidden_dim) = (2, 4)
x = np.array([
    [0.2, 0.5, 0.1, 0.7],
    [0.9, 0.3, 0.4, 0.2],
])

# 第一层把 4 维扩到 8 维。
W1 = np.random.randn(4, 8) * 0.3

# 第二层再把 8 维压回 4 维。
W2 = np.random.randn(8, 4) * 0.3

def relu(z):
    return np.maximum(z, 0)

# x @ W1 后 shape = (2, 8)
hidden = relu(x @ W1)

# hidden @ W2 后 shape 回到 (2, 4)
out = hidden @ W2

print('hidden.shape =', hidden.shape)
print('FFN output =')
print(out)


## 为什么现代模型喜欢更大的中间层

FFN 常常会先把维度扩到原来的 2 到 4 倍，原因并不神秘：

- 先扩宽，模型就有更大的中间表示空间
- 在更高维空间做非线性，可以表达更复杂的特征组合
- 再压回原维度，便于和 residual 路径继续对齐

所以 FFN 不是“凑个 MLP 进去”，而是在 block 里承担非常重要的逐位置非线性建模工作。


## SwiGLU 到底改了什么

SwiGLU 的关键不是“换一个激活函数”这么简单，而是引入了门控思想。

直觉上可以这样理解：

- 普通 FFN：所有扩展出来的特征都按同一种方式继续往后走
- SwiGLU：让一部分特征去控制另一部分特征该放大还是压低

所以它的味道更像：

**不是所有信息都无条件通过，而是先决定“哪些成分应该被放行得更多”。**


In [ ]:
W_gate = np.random.randn(4, 8) * 0.3
W_value = np.random.randn(4, 8) * 0.3
W_down = np.random.randn(8, 4) * 0.3

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def swish(z):
    return z * sigmoid(z)

# gate 分支负责生成“门”
gate = swish(x @ W_gate)

# value 分支负责生成被门控的内容
value = x @ W_value

# 两者逐元素相乘，就是门控过程
swiglu_hidden = gate * value

# 最后再压回原来的 hidden dimension
swiglu_out = swiglu_hidden @ W_down

print('gate.shape =', gate.shape)
print('value.shape =', value.shape)
print('swiglu_hidden.shape =', swiglu_hidden.shape)
print('swiglu_out =')
print(swiglu_out)


## 为什么门控有吸引力

门控的吸引力在于，它让模型不只是“生成一堆中间特征”，还可以进一步决定：

- 哪些特征应该被压低
- 哪些特征应该被放大
- 哪些特征更值得继续传下去

所以 SwiGLU 在工程上常被看成一种更灵活的逐位置非线性结构。

## Common mistakes

- 误以为 attention 已经足够表达一切，因此 FFN 可有可无。实际上 FFN 提供了重要的逐位置非线性容量。
- 只记得 FFN 是“两层 MLP”，却不理解它在每个 token 上独立工作。
- 把 SwiGLU 看成只是换了个激活函数，而没看到其中的门控机制。
- 以为 FFN 只是“补点参数”。实际上它承担的是另一类表达能力。

## Checkpoints

- 比较 ReLU FFN 和 SwiGLU FFN 的中间隐藏表示。
- 思考为什么 FFN 常常会先把维度扩到原来的 2 到 4 倍。
- 试着用自己的话解释“门控”意味着什么。
- 回答：attention 和 FFN 分别负责“跨位置处理”还是“单位置处理”？

## Summary

Transformer block 里，attention 负责跨位置交换信息，FFN 负责在每个位置内部加工信息。SwiGLU 则进一步引入门控，让这种逐位置加工更灵活。二者配合，才构成现代大模型的基本表达单元。

## Next Module

下一节把前面的 attention、norm、FFN 真正拼起来，构成一个完整的基础 Transformer block。
